In [1]:
%pip install yfinance

import yfinance as yf

ticker = yf.Ticker("m")

price = ticker.history(start="2009-02-14", end="2020-06-11")
print(price)

Note: you may need to restart the kernel to use updated packages.
                               Open      High       Low     Close     Volume  \
Date                                                                           
2009-02-17 00:00:00-05:00  4.532084  4.745755  4.487100  4.543329   13025600   
2009-02-18 00:00:00-05:00  4.593936  4.650166  4.340904  4.436494   12778100   
2009-02-19 00:00:00-05:00  4.560199  4.695150  4.273429  4.307167   12208200   
2009-02-20 00:00:00-05:00  4.245313  4.498345  4.132855  4.419624   16329100   
2009-02-23 00:00:00-05:00  4.498344  4.498344  4.132854  4.160968   15378400   
...                             ...       ...       ...       ...        ...   
2020-06-04 00:00:00-04:00  6.442634  7.057801  6.193242  6.825036   78157800   
2020-06-05 00:00:00-04:00  7.814293  7.980555  7.215752  7.290570   76114500   
2020-06-08 00:00:00-04:00  7.781041  7.955616  7.564902  7.938990   70444000   
2020-06-09 00:00:00-04:00  8.612347  8.695478  7.28225

In [2]:
#Collect Features, we want them a day behind, we give yesturdays stats to guess todays price, so score follows current day
price["Change"] = price["Close"].shift(1) - price["Open"].shift(1)
price["Score"] = (price["Close"] > price["Open"]).astype(int)
price["%Change"] = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
price["YestScore"] = price["Score"].shift(1)
price["5DayMean"] = price["Close"].rolling(5).mean().shift(1)
price["5DayVoli"] = price["Close"].rolling(5).std().shift(1)
price["5DayPerf"] = price["Score"].rolling(5).sum().shift(1)

price.dropna(inplace=True)

print(price["5DayPerf"])

Date
2009-02-24 00:00:00-05:00    2.0
2009-02-25 00:00:00-05:00    2.0
2009-02-26 00:00:00-05:00    2.0
2009-02-27 00:00:00-05:00    2.0
2009-03-02 00:00:00-05:00    2.0
                            ... 
2020-06-04 00:00:00-04:00    3.0
2020-06-05 00:00:00-04:00    4.0
2020-06-08 00:00:00-04:00    4.0
2020-06-09 00:00:00-04:00    4.0
2020-06-10 00:00:00-04:00    3.0
Name: 5DayPerf, Length: 2844, dtype: float64


In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]

priceTrain = price[price.index <= "2018-06-11"]
priceTest = price[price.index > "2018-06-11"]

results = []

for i in range(len(priceTest)):
    xTrain = priceTrain[features]
    yTrain = priceTrain["Score"]

    scaler = StandardScaler()
    xTrain = scaler.fit_transform(xTrain)

    model = LogisticRegression()
    model.fit(xTrain, yTrain)

    #Grab yesturdays information to predict todays, data already transformed so i is yesturday
    yest = priceTest.iloc[[i]]
    yestScal = scaler.transform(yest[features])
    pred = model.predict(yestScal)[0]
    prob = model.predict_proba(yestScal)[0][1]
    print("Analyzing Day: ", priceTest.index[i])

    results.append({"Date": priceTest.index[i],
                    "Score": priceTest["Score"][i],
                    "Prediction": pred,
                    "Probability": prob,
                    "Open": priceTest["Open"][i],
                    "Close": priceTest["Close"][i]})
    
    priceTrain = pd.concat([priceTrain, yest])

Analyzing Day:  2018-06-12 00:00:00-04:00
Analyzing Day:  2018-06-13 00:00:00-04:00
Analyzing Day:  2018-06-14 00:00:00-04:00
Analyzing Day:  2018-06-15 00:00:00-04:00
Analyzing Day:  2018-06-18 00:00:00-04:00
Analyzing Day:  2018-06-19 00:00:00-04:00
Analyzing Day:  2018-06-20 00:00:00-04:00
Analyzing Day:  2018-06-21 00:00:00-04:00
Analyzing Day:  2018-06-22 00:00:00-04:00
Analyzing Day:  2018-06-25 00:00:00-04:00
Analyzing Day:  2018-06-26 00:00:00-04:00
Analyzing Day:  2018-06-27 00:00:00-04:00
Analyzing Day:  2018-06-28 00:00:00-04:00
Analyzing Day:  2018-06-29 00:00:00-04:00
Analyzing Day:  2018-07-02 00:00:00-04:00
Analyzing Day:  2018-07-03 00:00:00-04:00
Analyzing Day:  2018-07-05 00:00:00-04:00
Analyzing Day:  2018-07-06 00:00:00-04:00
Analyzing Day:  2018-07-09 00:00:00-04:00
Analyzing Day:  2018-07-10 00:00:00-04:00
Analyzing Day:  2018-07-11 00:00:00-04:00
Analyzing Day:  2018-07-12 00:00:00-04:00
Analyzing Day:  2018-07-13 00:00:00-04:00
Analyzing Day:  2018-07-16 00:00:0

In [4]:
results = pd.DataFrame(results)

score = results["Score"]
pred = results["Prediction"]
op = results["Open"]
close = results["Close"]

accuracy = accuracy_score(score, pred)
precision = precision_score(score, pred, zero_division=0)
recall = recall_score(score, pred, zero_division=0)
f1 = f1_score(score, pred, zero_division=0)

print("Accuracy: ", accuracy, "\nPrecision: ", precision, "\nRecall: ", recall, "\nF1: ", f1)



Accuracy:  0.5149105367793241 
Precision:  0.4744186046511628 
Recall:  0.43776824034334766 
F1:  0.45535714285714285


In [5]:
#Find out how much you would have made/lost with this method
totalIn = 0
totalOut = 0
totalInvested = 0
totalStoodOut = 0
alltradein = 0
alltradeout = 0
perftradein = 0
perftradeout = 0
for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
    if i == 1:
        totalIn += 100
        totalInvested += 1
        temp = ((c - o)/o) * 100
        totalOut += 100 + temp
        print(temp, o, c)
    else:
        totalStoodOut += 1
    alltradein += 100
    alltradeout += (((c-o)/o) * 100) + 100
    if s == 1:
        perftradein += 100
        perftradeout += (((c-o)/o) * 100) + 100

print(totalIn)
print(totalOut)
print(totalInvested)
print(totalStoodOut)
print(alltradein)
print(alltradeout)
print(perftradein)
print(perftradeout)

-4.027015079504183 28.542826007029078 27.393402099609375
2.490627480919609 26.923290314781198 27.593849182128906
1.8674357599269276 27.413588783220135 27.925519943237305
1.540872652078289 27.60827728973661 28.0336856842041
-0.86364308888855 27.55059329013732 27.312654495239258
-3.7294257281504395 28.03367922062072 26.988183975219727
-0.7419633478764915 26.23831884565246 26.04364013671875
1.1484877964109443 26.36810780621964 26.670942306518555
1.9751804156599804 28.473514009734025 29.035917282104492
-1.7313894574713657 29.151284848884988 28.646562576293945
0.30903663720354185 27.997633092571085 28.084156036376953
0.9979513085348116 28.17788956885357 28.459091186523438
-9.686528315601329 28.062522465990117 25.34423828125
-0.1305804018413367 27.608276569164193 27.57222557067871
0.1101953618955096 26.17341336144938 26.202255249023438
-0.14044730113372378 25.938138017715495 25.901708602905273
-1.1242313246026319 25.92355899578006 25.632118225097656
-2.6610665059863434 26.010994634149668 25.